# 🤖 Notebook 06 — All ML Classifiers + Cross-Validation + Tuning

**Phases 8–12 of the project**

Classifiers trained:
1. Logistic Regression
2. k-NN
3. LDA
4. SVM
5. Decision Tree
6. Random Forest

Each uses: `Fused Features → StandardScaler → Classifier → Evaluate`

Also includes:
- StratifiedGroupKFold cross-validation (patient-aware)
- GridSearchCV hyperparameter tuning for top models
- Full metrics table

**Run each cell with Shift+Enter**

## 0. Setup & Imports

In [1]:
import sys
sys.path.insert(0, r'D:\Projects\ML')
import matplotlib
matplotlib.use('Agg')


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              matthews_corrcoef, ConfusionMatrixDisplay, RocCurveDisplay)

from src.config import FEATURES_FUSED_CSV, FIGURES_DIR, MODELS_DIR, RESULTS_DIR, RANDOM_SEED
from src.evaluation import compute_all_metrics, plot_confusion_matrix, print_results_table
from src.utils import set_plot_style, save_figure

set_plot_style()
np.random.seed(RANDOM_SEED)
print('Imports done!')

Imports done!


## 1. Load Fused Features & Prepare Data

In [2]:
df = pd.read_csv(FEATURES_FUSED_CSV)
feat_cols = [c for c in df.columns if c.startswith('glcm_') or c.startswith('lbp_')]

# Split by pre-assigned split column
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']
test_df  = df[df['split'] == 'test']

X_train = train_df[feat_cols].values.astype(np.float32)
y_train = train_df['label'].values
groups_train = train_df['patient_id'].values

X_val   = val_df[feat_cols].values.astype(np.float32)
y_val   = val_df['label'].values

X_test  = test_df[feat_cols].values.astype(np.float32)
y_test  = test_df['label'].values

print(f'Features      : {len(feat_cols)}')
print(f'Train samples : {X_train.shape[0]}  (patients: {train_df["patient_id"].nunique()})')
print(f'Val   samples : {X_val.shape[0]}   (patients: {val_df["patient_id"].nunique()})')
print(f'Test  samples : {X_test.shape[0]}   (patients: {test_df["patient_id"].nunique()})')

Features      : 72
Train samples : 5403  (patients: 57)
Val   samples : 1171   (patients: 12)
Test  samples : 1335   (patients: 13)


## 2. Define All 6 Classifiers (with Pipelines)

In [3]:
# Each pipeline: StandardScaler -> Classifier
# Scaler is fit ONLY on training data (no leakage)

classifiers = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
    ]),
    'k-NN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'LDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis())
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=RANDOM_SEED))
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=RANDOM_SEED))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    ]),
}

print(f'Classifiers ready: {list(classifiers.keys())}')

Classifiers ready: ['Logistic Regression', 'k-NN', 'LDA', 'SVM', 'Decision Tree', 'Random Forest']


## 3. Patient-Aware Cross-Validation (StratifiedGroupKFold)

In [4]:
print('Running 5-Fold Patient-Aware Cross-Validation...')
print('(Each fold keeps all images from a patient in the same set)\n')

cv = StratifiedGroupKFold(n_splits=5)
cv_results = {}

for name, pipeline in classifiers.items():
    scores = cross_val_score(
        pipeline, X_train, y_train,
        groups=groups_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )
    cv_results[name] = scores
    print(f'  {name:22s}: F1 = {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)')

print('\nCross-validation complete!')

Running 5-Fold Patient-Aware Cross-Validation...
(Each fold keeps all images from a patient in the same set)

  Logistic Regression   : F1 = 83.42% (+/- 3.05%)
  k-NN                  : F1 = 78.16% (+/- 1.58%)
  LDA                   : F1 = 84.01% (+/- 2.76%)
  SVM                   : F1 = 84.16% (+/- 1.43%)
  Decision Tree         : F1 = 75.66% (+/- 0.86%)
  Random Forest         : F1 = 82.80% (+/- 0.98%)

Cross-validation complete!


In [5]:
# Plot CV results
fig, ax = plt.subplots(figsize=(12, 5))
names  = list(cv_results.keys())
means  = [cv_results[n].mean()*100 for n in names]
stds   = [cv_results[n].std()*100  for n in names]
colors = ['#7c3aed','#2563eb','#059669','#d97706','#dc2626','#db2777']

bars = ax.bar(names, means, yerr=stds, color=colors,
              capsize=5, edgecolor='#3d3f4e', linewidth=1.5)
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, m+0.5,
            f'{m:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_title('5-Fold Cross-Validation F1 Scores (Patient-Aware)', fontsize=13)
ax.set_ylabel('F1 Score (%)')
ax.set_ylim(0, 110)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
save_figure(fig, FIGURES_DIR / '06_cv_scores.png')
plt.show()

📊  Saved figure → D:\Projects\ML\outputs\figures\06_cv_scores.png


## 4. Hyperparameter Tuning — Top Models

In [6]:
print('Tuning SVM...')
svm_params = {
    'clf__C'     : [0.1, 1, 10, 100],
    'clf__gamma' : ['scale', 'auto'],
    'clf__kernel': ['rbf', 'linear'],
}
svm_grid = GridSearchCV(
    classifiers['SVM'], svm_params,
    cv=StratifiedGroupKFold(n_splits=3),
    scoring='f1', n_jobs=-1, refit=True
)
svm_grid.fit(X_train, y_train, groups=groups_train)
print(f'Best SVM params : {svm_grid.best_params_}')
print(f'Best SVM CV F1  : {svm_grid.best_score_*100:.2f}%')
classifiers['SVM'] = svm_grid.best_estimator_

Tuning SVM...
Best SVM params : {'clf__C': 0.1, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
Best SVM CV F1  : 83.53%


In [7]:
print('Tuning Random Forest...')
rf_params = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth'   : [None, 10, 20],
    'clf__min_samples_split': [2, 5],
}
rf_grid = GridSearchCV(
    classifiers['Random Forest'], rf_params,
    cv=StratifiedGroupKFold(n_splits=3),
    scoring='f1', n_jobs=-1, refit=True
)
rf_grid.fit(X_train, y_train, groups=groups_train)
print(f'Best RF params : {rf_grid.best_params_}')
print(f'Best RF CV F1  : {rf_grid.best_score_*100:.2f}%')
classifiers['Random Forest'] = rf_grid.best_estimator_

Tuning Random Forest...
Best RF params : {'clf__max_depth': 10, 'clf__min_samples_split': 2, 'clf__n_estimators': 100}
Best RF CV F1  : 83.06%


In [8]:
print('Tuning k-NN...')
knn_params = {
    'clf__n_neighbors': [3, 5, 7, 11, 15],
    'clf__metric'     : ['euclidean', 'manhattan'],
}
knn_grid = GridSearchCV(
    classifiers['k-NN'], knn_params,
    cv=StratifiedGroupKFold(n_splits=3),
    scoring='f1', n_jobs=-1, refit=True
)
knn_grid.fit(X_train, y_train, groups=groups_train)
print(f'Best k-NN params : {knn_grid.best_params_}')
print(f'Best k-NN CV F1  : {knn_grid.best_score_*100:.2f}%')
classifiers['k-NN'] = knn_grid.best_estimator_

Tuning k-NN...
Best k-NN params : {'clf__metric': 'euclidean', 'clf__n_neighbors': 15}
Best k-NN CV F1  : 78.83%


## 5. Train All Classifiers & Evaluate on Validation Set

In [9]:
print('Training all classifiers on full training set...')
val_results = []
trained_models = {}
roc_data = []

for name, pipeline in classifiers.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    y_prob = pipeline.predict_proba(X_val)[:,1] if hasattr(pipeline, 'predict_proba') else None

    metrics = compute_all_metrics(y_val, y_pred, y_prob, name)
    val_results.append(metrics)
    trained_models[name] = pipeline

    if y_prob is not None:
        roc_data.append({'name': name, 'y_true': y_val, 'y_prob': y_prob})

    print(f'  {name:22s}: Acc={metrics["accuracy"]}%  F1={metrics["f1"]}%  AUC={metrics["roc_auc"]}%')

results_df = print_results_table(val_results)
results_df.to_csv(RESULTS_DIR / 'validation_results.csv', index=False)

Training all classifiers on full training set...
  Logistic Regression   : Acc=78.74%  F1=86.76%  AUC=70.85%
  k-NN                  : Acc=74.81%  F1=84.08%  AUC=65.03%
  LDA                   : Acc=78.48%  F1=86.57%  AUC=71.55%
  SVM                   : Acc=81.13%  F1=89.19%  AUC=64.25%
  Decision Tree         : Acc=69.09%  F1=79.24%  AUC=60.71%
  Random Forest         : Acc=78.99%  F1=87.58%  AUC=59.45%

📊  RESULTS TABLE
              model  accuracy  precision  recall  specificity    f1  roc_auc    mcc
Logistic Regression     78.74      84.30   89.38        41.09 86.76    70.85 0.3335
               k-NN     74.81      82.87   85.32        37.60 84.08    65.03 0.2387
                LDA     78.48      84.32   88.94        41.47 86.57    71.55 0.3298
                SVM     81.13      80.57   99.89        14.73 89.19    64.25 0.3377
      Decision Tree     69.09      83.15   75.68        45.74 79.24    60.71 0.1956
      Random Forest     78.99      81.26   94.96        22.48 87.58  

## 6. ROC Curves — All Classifiers

In [10]:
colors = ['#7c3aed','#2563eb','#059669','#d97706','#dc2626','#db2777']
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0f1117')

for i, res in enumerate(roc_data):
    RocCurveDisplay.from_predictions(
        res['y_true'], res['y_prob'],
        name=res['name'], ax=ax, color=colors[i]
    )

ax.plot([0,1],[0,1],'w--', alpha=0.4, label='Random')
ax.set_title('ROC Curves — All Classifiers (Validation Set)', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
save_figure(fig, FIGURES_DIR / '06_roc_curves.png')
plt.show()

TypeError: RocCurveDisplay.from_predictions() got an unexpected keyword argument 'color'

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Confusion Matrices — Validation Set', fontsize=14)
axes = axes.flatten()

for i, (name, pipeline) in enumerate(trained_models.items()):
    y_pred = pipeline.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign','Malignant'],
                yticklabels=['Benign','Malignant'],
                ax=axes[i], cbar=False,
                annot_kws={'size':14,'weight':'bold'})
    axes[i].set_title(name, fontsize=11)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
save_figure(fig, FIGURES_DIR / '06_confusion_matrices.png')
plt.show()

## 8. Save All Trained Models

In [ ]:
for name, pipeline in trained_models.items():
    fname = name.lower().replace(' ','_').replace('-','') + '.pkl'
    joblib.dump(pipeline, MODELS_DIR / fname)
    print(f'Saved: {fname}')

print('\nAll models saved to outputs/models/')

## 9. Summary

In [ ]:
best = results_df.loc[results_df['f1'].idxmax()]
print('='*60)
print('  PHASES 8-12 COMPLETE - Classifiers + CV + Tuning')
print('='*60)
print(f'  Best model on validation: {best["model"]}')
print(f'  Accuracy  : {best["accuracy"]}%')
print(f'  F1-Score  : {best["f1"]}%')
print(f'  ROC-AUC   : {best["roc_auc"]}%')
print()
print('  Next: Notebook 07 - Final Test Evaluation + XAI')
print('='*60)